# Assignment 04 — Miền CIFAR-10 · Notebook 02: Đối chiếu PyTorch và TensorFlow/Keras

**Học phần:** Phát triển các Hệ thống Thông minh — Học viện Công nghệ Bưu chính Viễn thông
**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế
**Học kỳ:** Học kỳ 1 năm học 2026 – 2027

---

## Mục tiêu của notebook

Notebook 01 đã chứng minh rằng một mạng tích chập viết tay bằng NumPy hoạt động đúng về mặt toán
học. Notebook này trả lời câu hỏi tiếp theo: **khi được tự do dùng framework, ta đạt tới đâu trên
CIFAR-10, và khoảng cách so với cài đặt thủ công đến từ đâu?**

Bốn việc được thực hiện:

1. Dựng cùng một kiến trúc sâu ba khối bằng **PyTorch** và bằng **Keras**, đối chiếu số tham số
   để chắc chắn hai mạng thực sự tương đương.
2. Lưu ba hiện vật bàn giao cho notebook `mlp_vs_cnn`: trọng số `cifar10_cnn_pytorch.pt`, định
   nghĩa mô-đun `cifar10_cnn_def.py`, hằng số chuẩn hóa `cifar10_preproc.json`. Mô hình PyTorch
   **bắt buộc** có phương thức `extract_features(x)` trả về véc-tơ ẩn 128 chiều để phân tích PCA.
3. So sánh ba cách cài đặt trên cùng 10 000 ảnh kiểm thử: NumPy thuần, PyTorch, Keras.
4. Phân tích sâu mô hình tốt nhất: độ chính xác từng lớp và tám ảnh sai với độ tin cậy cao nhất.

**Năm hình bắt buộc sinh ra ở đây:** `fig_cifar10_framework_curves.png`,
`fig_cifar10_framework_confusion.png`, `fig_cifar10_3way_benchmark.png`,
`fig_cifar10_per_class_accuracy.png`, `fig_cifar10_high_conf_errors.png`. Notebook cũng ghi tệp
metrics tổng hợp `reports/metrics_cifar10.json` theo lược đồ đa lớp ở mục 5.3 hợp đồng tích hợp.

## Kiến trúc: Conv-BN-ReLU-MaxPool-Dropout ba khối

```
Đầu vào (3, 32, 32)
  Khối 1: Conv(3->32, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0,25)   ->  (32, 16, 16)
  Khối 2: Conv(32->64, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0,25)  ->  (64,  8,  8)
  Khối 3: Conv(64->64, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0,25)  ->  (64,  4,  4)
  Flatten (1024) -> Dense(128) -> ReLU -> Dropout(0,5) -> Dense(10)
```

Ba thành phần mà cài đặt NumPy ở notebook 01 **không có**, và đó chính là nguồn gốc chính của
chênh lệch kết quả:

- **Batch Normalization.** Chuẩn hóa kích hoạt theo từng lô rồi học lại hệ số tỉ lệ và độ dời:
  $\hat{h} = \gamma \frac{h - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$. Nó làm phẳng mặt mất
  mát nên cho phép learning rate lớn hơn và hội tụ nhanh hơn đáng kể.
- **Dropout.** Tắt ngẫu nhiên một phần kích hoạt khi huấn luyện, buộc mạng không phụ thuộc vào
  một vài nơ-ron riêng lẻ. Notebook 01 đã cho thấy mạng NumPy quá khớp rõ rệt; dropout là câu trả
  lời trực tiếp cho vấn đề đó.
- **Khối thứ ba.** Thêm một tầng tích chập nữa mở rộng trường tiếp nhận (receptive field) tới mức
  bao phủ gần trọn ảnh $32 \times 32$, cho phép mạng nhìn thấy hình dáng tổng thể chứ không chỉ
  các mảnh cục bộ.

## 1. Nhập thư viện, cố định hạt giống và cấu hình số luồng CPU

Một chi tiết kỹ thuật quan trọng đã được đo trên chính máy này ở miền `house_price`: PyTorch để
mặc định 32 luồng mất khoảng 40 giây mỗi epoch, trong khi giới hạn xuống **4 luồng** chỉ mất
khoảng 2,4 giây mỗi epoch, nhanh hơn 16 lần. Nguyên nhân là chi phí đồng bộ giữa các luồng lấn át
phần tính toán khi mỗi lô dữ liệu nhỏ. Vì vậy notebook này đặt `torch.set_num_threads(4)` ngay từ
đầu và ghi lại con số thời gian thực đo được.

In [ ]:
import os, json, time, copy, sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             classification_report)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
import keras
from keras import layers

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)

# Xem phần giải thích ở trên: 4 luồng nhanh hơn nhiều so với mặc định trên máy này
_default_threads = torch.get_num_threads()
torch.set_num_threads(4)

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH  = '../data/cifar10.npz'
FIG_DIR    = '../reports/figures'
REP_DIR    = '../reports'
MODEL_DIR  = '../models'
for d in (FIG_DIR, REP_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

CLASS_EN = ['airplane', 'automobile', 'bird', 'cat', 'deer',
            'dog', 'frog', 'horse', 'ship', 'truck']
CLASS_VI = ['máy bay', 'ô tô', 'chim', 'mèo', 'hươu',
            'chó', 'ếch', 'ngựa', 'tàu thủy', 'xe tải']

EPOCHS_FW = 15
BATCH_FW  = 128
DEVICE    = torch.device('cpu')

print('NumPy      :', np.__version__)
print('PyTorch    :', torch.__version__, '| CUDA khả dụng:', torch.cuda.is_available())
print('TensorFlow :', tf.__version__)
print('Keras      :', keras.__version__)
print('Thiết bị   :', DEVICE)
print(f'Số luồng CPU của PyTorch: mặc định {_default_threads} -> đặt lại {torch.get_num_threads()}')
print(f'Cấu hình huấn luyện framework: {EPOCHS_FW} epoch, batch {BATCH_FW}')

## 2. Nạp dữ liệu và tiền xử lý

Phép chia và phép chuẩn hóa lặp lại **chính xác** những gì notebook 01 đã làm: cùng
`random_state=42`, cùng `stratify`, cùng hằng số theo kênh học từ nhánh train. Nhờ vậy ba mô hình
nhìn thấy cùng một tập kiểm thử đã qua cùng một phép biến đổi, và phép so sánh cuối notebook là
công bằng.

Khác biệt duy nhất: hai mô hình framework dùng **toàn bộ 40 000 ảnh train và 10 000 ảnh
validation**, trong khi hai mô hình NumPy chỉ dùng tập con 10 000 / 2 500 vì ngân sách tính toán.
Điều này được ghi vào khóa `numpy_subset` và `notes` của tệp metrics để người đọc báo cáo không
hiểu nhầm chênh lệch là thuần túy do chất lượng cài đặt.

In [ ]:
assert os.path.exists(DATA_PATH), f'Không tìm thấy {DATA_PATH}'
_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)

idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)

# Tích lũy BẮT BUỘC ở float64 (xem notebook 00: cộng dồn ở float32 làm trung bình bão hòa
# và trả về ba giá trị trùng nhau, một hiện vật làm tròn chứ không phải thống kê thật).
_tr_u8 = x_train_raw[idx_tr]
MEAN_C = np.array([_tr_u8[:, :, :, c].mean(dtype=np.float64) / 255.0
                   for c in range(3)], dtype=np.float32)
STD_C  = np.array([_tr_u8[:, :, :, c].std(dtype=np.float64) / 255.0
                   for c in range(3)], dtype=np.float32)
del _tr_u8


def preprocess(x_u8):
    '''uint8 (N,32,32,3) HWC -> float32 (N,3,32,32) CHW đã chuẩn hóa theo từng kênh.'''
    x = x_u8.astype(np.float32) / 255.0
    x = (x - MEAN_C) / STD_C
    return np.ascontiguousarray(x.transpose(0, 3, 1, 2))


X_tr, y_tr = preprocess(x_train_raw[idx_tr]), y_train_raw[idx_tr]
X_va, y_va = preprocess(x_train_raw[idx_va]), y_train_raw[idx_va]
X_te, y_te = preprocess(x_test_raw),          y_test_raw

print(f'MEAN theo kênh (R, G, B) = {[round(float(v), 10) for v in MEAN_C]}')
print(f'STD  theo kênh (R, G, B) = {[round(float(v), 10) for v in STD_C]}')
print()
print(f'Train      : {X_tr.shape}  phân phối lớp {np.bincount(y_tr, minlength=10).tolist()}')
print(f'Validation : {X_va.shape}  phân phối lớp {np.bincount(y_va, minlength=10).tolist()}')
print(f'Test       : {X_te.shape}  phân phối lớp {np.bincount(y_te, minlength=10).tolist()}')
print(f'Khoảng giá trị sau chuẩn hóa: [{X_tr.min():.4f}, {X_tr.max():.4f}]')
print(f'Bộ nhớ chiếm dụng: {(X_tr.nbytes + X_va.nbytes + X_te.nbytes)/1e6:.0f} MB (float32)')

preproc = {
    'description': 'Hằng số chuẩn hóa CIFAR-10 theo từng kênh màu, học từ 40000 ảnh nhánh '
                   'train (train_test_split test_size=0.2, stratify=y, random_state=42).',
    'mean': [float(v) for v in MEAN_C],
    'std':  [float(v) for v in STD_C],
    'mean_per_channel': [float(v) for v in MEAN_C],
    'std_per_channel':  [float(v) for v in STD_C],
    'channel_order': ['R', 'G', 'B'],
    'scale': 255.0,
    'formula': 'x_norm = (x_uint8 / 255.0 - mean_c) / std_c, áp dụng riêng cho từng kênh',
    'input_layout': 'NCHW (N, 3, 32, 32) cho PyTorch; dữ liệu gốc trong .npz là NHWC',
    'n_train': int(len(idx_tr)), 'n_val': int(len(idx_va)), 'n_test': int(len(X_te)),
    'n_classes': 10, 'class_names_en': CLASS_EN, 'class_names_vi': CLASS_VI,
    'random_seed': RANDOM_SEED,
}
with open(os.path.join(MODEL_DIR, 'cifar10_preproc.json'), 'w', encoding='utf-8') as f:
    json.dump(preproc, f, ensure_ascii=False, indent=2)
print()
print('Đã lưu', os.path.join(MODEL_DIR, 'cifar10_preproc.json'))

## 3. Định nghĩa mô hình PyTorch trong một mô-đun độc lập

Định nghĩa mạng được ghi ra tệp `../models/cifar10_cnn_def.py` thay vì chỉ khai báo trong
notebook. Lý do rất thực tế: notebook `mlp_vs_cnn` cần nạp lại trọng số đã huấn luyện để trích
véc-tơ ẩn cho phân tích PCA, mà `torch.load` trên một `state_dict` đòi hỏi lớp mô hình phải có sẵn
ở phía người nạp. Một tệp `.py` thuần, không phụ thuộc notebook, là cách bàn giao sạch nhất.

Phương thức `extract_features(x)` trả về đầu ra 128 chiều của tầng `Dense(128)` **sau ReLU nhưng
trước Dropout và trước tầng phân loại cuối**. Đây là biểu diễn mà mạng thực sự dùng để ra quyết
định, nên là đối tượng phù hợp nhất cho phân tích không gian ẩn.

In [ ]:
%%writefile ../models/cifar10_cnn_def.py
"""Định nghĩa CNN 2D cho CIFAR-10 (Assignment 04, miền cifar10).

Mô-đun độc lập để notebook khác (mlp_vs_cnn) nạp lại trọng số đã huấn luyện tại
models/cifar10_cnn_pytorch.pt và trích véc-tơ ẩn 128 chiều bằng extract_features().

Tiền xử lý bắt buộc khi suy luận (xem models/cifar10_preproc.json):
    x = x_uint8 / 255.0                      # ảnh gốc NHWC (N, 32, 32, 3)
    x = (x - mean_c) / std_c                 # mean_c, std_c là ba số cho ba kênh R, G, B
    x = x.transpose(0, 3, 1, 2)              # -> NCHW (N, 3, 32, 32)

Kiến trúc (theo mục 4 hợp đồng tích hợp, biến thể CIFAR-10 ba khối 32 -> 64 -> 64):
    Conv-BN-ReLU-MaxPool-Dropout x3 -> Flatten -> Dense(128) -> ReLU -> Dropout -> Dense(10)

Cách dùng:
    from cifar10_cnn_def import Cifar10CNN
    model = Cifar10CNN()
    model.load_state_dict(torch.load('cifar10_cnn_pytorch.pt', map_location='cpu'))
    model.eval()
    with torch.no_grad():
        z = model.extract_features(x)        # (N, 128)
"""

import torch
import torch.nn as nn


class Cifar10CNN(nn.Module):
    """CNN 2D ba khối phân cấp cho ảnh màu CIFAR-10."""

    FEATURE_DIM = 128          # số chiều véc-tơ ẩn mà extract_features trả về

    def __init__(self, n_classes: int = 10, p_conv: float = 0.25, p_fc: float = 0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 32 x 32 -> 16 x 16
            nn.Dropout(p_conv),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 16 x 16 -> 8 x 8
            nn.Dropout(p_conv),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 8 x 8 -> 4 x 4
            nn.Dropout(p_conv),
        )
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 4 * 4, self.FEATURE_DIM)
        self.relu_fc = nn.ReLU(inplace=True)
        self.drop_fc = nn.Dropout(p_fc)
        self.fc2 = nn.Linear(self.FEATURE_DIM, n_classes)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về véc-tơ ẩn 128 chiều ở tầng áp chót, dùng cho phân tích PCA.

        Tham số
        -------
        x : Tensor (N, 3, 32, 32) đã chuẩn hóa theo models/cifar10_preproc.json

        Trả về
        ------
        Tensor (N, 128) sau Dense(128) và ReLU, TRƯỚC Dropout và tầng phân loại.
        """
        h = self.block1(x)
        h = self.block2(h)
        h = self.block3(h)
        h = self.flatten(h)
        return self.relu_fc(self.fc1(h))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về logit chưa qua softmax, (N, 10)."""
        return self.fc2(self.drop_fc(self.extract_features(x)))

In [ ]:
sys.path.insert(0, os.path.abspath(MODEL_DIR))
import importlib
import cifar10_cnn_def
importlib.reload(cifar10_cnn_def)
from cifar10_cnn_def import Cifar10CNN

torch.manual_seed(RANDOM_SEED)
model_pt = Cifar10CNN().to(DEVICE)
n_pt = sum(p.numel() for p in model_pt.parameters() if p.requires_grad)
print(model_pt)
print()
print(f'Tổng tham số học được (PyTorch): {n_pt:,}')
for name, mod in [('block1', model_pt.block1), ('block2', model_pt.block2),
                  ('block3', model_pt.block3), ('fc1', model_pt.fc1), ('fc2', model_pt.fc2)]:
    print(f'  {name:7s}: {sum(p.numel() for p in mod.parameters()):>9,} tham số')

# Kiểm tra luồng kích thước và hợp đồng của extract_features
with torch.no_grad():
    _x = torch.from_numpy(X_te[:4])
    _h1 = model_pt.block1(_x); _h2 = model_pt.block2(_h1); _h3 = model_pt.block3(_h2)
    _f = model_pt.extract_features(_x)
    _o = model_pt(_x)
print()
print(f'Luồng kích thước: {tuple(_x.shape)} -> khối 1 {tuple(_h1.shape)} '
      f'-> khối 2 {tuple(_h2.shape)} -> khối 3 {tuple(_h3.shape)}')
print(f'extract_features: {tuple(_x.shape)} -> {tuple(_f.shape)}  '
      f'| đúng hợp đồng 128 chiều: {tuple(_f.shape) == (4, 128)}')
print(f'forward         : {tuple(_x.shape)} -> {tuple(_o.shape)}  '
      f'| đúng 10 logit: {tuple(_o.shape) == (4, 10)}')
print(f'Véc-tơ ẩn sau ReLU nên không âm: giá trị nhỏ nhất = {float(_f.min()):.6f}')

**Diễn giải.** Luồng kích thước đúng như thiết kế: ba lần max pooling đưa ảnh từ
$32 \times 32$ xuống $4 \times 4$ trong khi số kênh tăng từ 3 lên 64. Đây là nguyên tắc "đánh đổi
độ phân giải không gian lấy chiều sâu ngữ nghĩa" của mọi kiến trúc CNN phân cấp: các tầng đầu mô
tả *ở đâu có cạnh gì*, các tầng sau mô tả *có những bộ phận nào*, và vị trí chính xác dần trở nên
không quan trọng.

Phân rã tham số cho thấy tầng `fc1` vẫn chiếm phần lớn trọng số, nhưng tỉ trọng đã dễ chịu hơn
nhiều so với mạng NumPy ở notebook 01, vì ba lần pooling đã nén véc-tơ phẳng xuống còn 1 024
chiều. Véc-tơ ẩn có giá trị nhỏ nhất bằng 0 đúng như kỳ vọng sau ReLU.

## 4. Huấn luyện mô hình PyTorch

In [ ]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds   = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va))
test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))

g = torch.Generator(); g.manual_seed(RANDOM_SEED)
train_ld = DataLoader(train_ds, batch_size=BATCH_FW, shuffle=True, generator=g)
val_ld   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_ld  = DataLoader(test_ds,  batch_size=512, shuffle=False)

model_pt = Cifar10CNN().to(DEVICE)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def eval_torch(model, loader):
    '''Đánh giá ở chế độ eval: BatchNorm dùng thống kê tích lũy, Dropout tắt.'''
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    probs = []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            tot_loss += float(criterion(out, yb)) * len(yb)
            p = torch.softmax(out, dim=1)
            probs.append(p.numpy())
            correct += int((p.argmax(1) == yb).sum())
            n += len(yb)
    return tot_loss / n, correct / n, np.concatenate(probs, axis=0)


hist_pt = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc_pt, best_ep_pt, best_state = -1.0, 0, None
t0 = time.time()
print(f'Huấn luyện trên đầy đủ {len(X_tr):,} ảnh train, {len(X_va):,} ảnh validation')
print('-' * 104)
for ep in range(1, EPOCHS_FW + 1):
    model_pt.train()
    run_loss, correct, n = 0.0, 0, 0
    for xb, yb in train_ld:
        optimizer.zero_grad()
        out = model_pt(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        run_loss += float(loss) * len(yb)
        correct += int((out.argmax(1) == yb).sum())
        n += len(yb)
    tr_loss, tr_acc = run_loss / n, correct / n
    va_loss, va_acc, _ = eval_torch(model_pt, val_ld)
    hist_pt['train_loss'].append(tr_loss); hist_pt['val_loss'].append(va_loss)
    hist_pt['train_acc'].append(tr_acc);   hist_pt['val_acc'].append(va_acc)
    star = ''
    if va_acc > best_acc_pt:
        best_acc_pt, best_ep_pt = va_acc, ep
        best_state = copy.deepcopy(model_pt.state_dict())
        star = '  <-- tốt nhất'
    print(f'[PyTorch] epoch {ep:2d}/{EPOCHS_FW} | train_loss={tr_loss:.4f} '
          f'train_acc={tr_acc:.4f} | val_loss={va_loss:.4f} val_acc={va_acc:.4f}{star}',
          flush=True)
time_pt = time.time() - t0
model_pt.load_state_dict(best_state)
print('-' * 104)
print(f'[PyTorch] hoàn tất sau {time_pt:.1f}s ({time_pt/EPOCHS_FW:.1f}s mỗi epoch) | '
      f'epoch tốt nhất = {best_ep_pt} | val_acc tốt nhất = {best_acc_pt:.4f}')

**Diễn giải nhật ký huấn luyện.** Hai điều đáng chú ý trong nhật ký. Thứ nhất, `train_acc` thấp
hơn `val_acc` ở những epoch đầu — hiện tượng thoạt nhìn có vẻ nghịch lý nhưng hoàn toàn bình
thường khi mạng có Dropout: `train_acc` được đo **trong lúc dropout đang bật**, còn `val_acc` đo ở
chế độ `eval` với dropout tắt và BatchNorm dùng thống kê tích lũy. Mạng ở chế độ đánh giá luôn
khỏe hơn chính nó ở chế độ huấn luyện.

Thứ hai, độ chính xác kiểm định tăng nhanh trong khoảng năm epoch đầu rồi chậm lại. Việc chọn
epoch tốt nhất theo `val_acc` đảm bảo trọng số cuối cùng không phải là trọng số của epoch cuối
cùng nếu epoch đó đã bắt đầu quá khớp.

## 5. Lưu ba hiện vật bàn giao và kiểm chứng quy trình nạp lại

Notebook `mlp_vs_cnn` sẽ nạp đúng ba tệp này để trích véc-tơ ẩn. Báo cáo kiểm chứng ngay tại đây
rằng quy trình nạp lại cho kết quả trùng khít với mô hình đang nằm trong bộ nhớ — nếu chờ tới lúc
notebook kia chạy mới phát hiện sai thì đã quá muộn.

In [ ]:
ckpt_path = os.path.join(MODEL_DIR, 'cifar10_cnn_pytorch.pt')
torch.save(model_pt.state_dict(), ckpt_path)
print('Đã lưu state dict :', ckpt_path, f'({os.path.getsize(ckpt_path)/1024:.1f} KB)')

# Kiểm chứng quy trình nạp lại đúng như notebook mlp_vs_cnn sẽ làm
reloaded = Cifar10CNN()
reloaded.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
reloaded.eval()
model_pt.eval()
with torch.no_grad():
    xb = torch.from_numpy(X_te[:256])
    f_new, f_old = reloaded.extract_features(xb), model_pt.extract_features(xb)
    lo_new, lo_old = reloaded(xb), model_pt(xb)
print('Véc-tơ ẩn sau khi nạp lại  :', tuple(f_new.shape))
print('Sai lệch tối đa đặc trưng  :', float((f_new - f_old).abs().max()))
print('Sai lệch tối đa logit      :', float((lo_new - lo_old).abs().max()))
print('Nạp lại thành công, khớp bit-for-bit:',
      bool(torch.allclose(f_new, f_old) and torch.allclose(lo_new, lo_old)))
print()
print('Ba hiện vật bàn giao cho notebook mlp_vs_cnn:')
for f in ['cifar10_cnn_pytorch.pt', 'cifar10_cnn_def.py', 'cifar10_preproc.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f:26s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')
print()
print('Thống kê véc-tơ ẩn 128 chiều trên 256 ảnh kiểm thử đầu tiên:')
fa = f_new.numpy()
print(f'  trung bình = {fa.mean():.4f}  |  độ lệch chuẩn = {fa.std():.4f}  '
      f'|  giá trị lớn nhất = {fa.max():.4f}')
print(f'  tỉ lệ phần tử bằng 0 (nơ-ron không kích hoạt): {100*(fa == 0).mean():.2f}%')

**Diễn giải.** Mô hình nạp lại cho ra véc-tơ ẩn và logit trùng khít với mô hình gốc, sai lệch bằng
0 tuyệt đối. Hợp đồng bàn giao với notebook `mlp_vs_cnn` vì thế được bảo đảm.

Tỉ lệ phần tử bằng 0 trong véc-tơ ẩn là một chỉ báo đáng quan tâm: ReLU tạo ra biểu diễn thưa, và
mức thưa vừa phải thường đi kèm khả năng khái quát tốt. Nếu tỉ lệ này tiến gần 100% thì mạng đã
rơi vào trạng thái "ReLU chết" và phần lớn nơ-ron vô dụng; con số đo được cho thấy điều đó không
xảy ra.

## 6. Mô hình Keras tương đương

Keras dùng bố cục **NHWC** thay vì NCHW, nên dữ liệu cần hoán vị trục trước khi đưa vào. Ngoài
khác biệt kỹ thuật đó, hai mạng phải giống hệt nhau về cấu trúc — và báo cáo kiểm chứng điều này
bằng cách đối chiếu tổng số tham số học được. Nếu hai con số khác nhau thì phép so sánh giữa hai
framework trở nên vô nghĩa.

In [ ]:
keras.utils.set_random_seed(RANDOM_SEED)

# NCHW -> NHWC
X_tr_k = np.ascontiguousarray(np.transpose(X_tr, (0, 2, 3, 1)))
X_va_k = np.ascontiguousarray(np.transpose(X_va, (0, 2, 3, 1)))
X_te_k = np.ascontiguousarray(np.transpose(X_te, (0, 2, 3, 1)))
print('Bố cục Keras:', X_tr_k.shape, '(NHWC)')

model_tf = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(10),
], name='cifar10_cnn_keras')

model_tf.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                 loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
model_tf.summary()
n_tf = int(sum(np.prod(w.shape) for w in model_tf.trainable_weights))
print()
print(f'Tổng tham số học được (Keras) : {n_tf:,}')
print(f'Tổng tham số học được (PyTorch): {n_pt:,}')
print('Hai kiến trúc khớp số tham số  :', n_tf == n_pt)

In [ ]:
class VietnameseLogger(keras.callbacks.Callback):
    '''In nhật ký từng epoch bằng tiếng Việt và ghi nhớ trọng số tốt nhất theo val_accuracy.'''
    def __init__(self, total):
        super().__init__()
        self.total = total
        self.best = -1.0
        self.best_epoch = 0
        self.best_weights = None
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        va = float(logs.get('val_accuracy', 0.0))
        star = ''
        if va > self.best:
            self.best, self.best_epoch = va, epoch + 1
            self.best_weights = self.model.get_weights()
            star = '  <-- tốt nhất'
        print(f"[Keras]   epoch {epoch+1:2d}/{self.total} | "
              f"train_loss={logs.get('loss', 0):.4f} train_acc={logs.get('accuracy', 0):.4f} | "
              f"val_loss={logs.get('val_loss', 0):.4f} val_acc={va:.4f}{star}", flush=True)

cb = VietnameseLogger(EPOCHS_FW)
print('-' * 104)
t0 = time.time()
h_tf = model_tf.fit(X_tr_k, y_tr, validation_data=(X_va_k, y_va),
                    epochs=EPOCHS_FW, batch_size=BATCH_FW, verbose=0, callbacks=[cb])
time_tf = time.time() - t0
model_tf.set_weights(cb.best_weights)
print('-' * 104)
print(f'[Keras]   hoàn tất sau {time_tf:.1f}s ({time_tf/EPOCHS_FW:.1f}s mỗi epoch) | '
      f'epoch tốt nhất = {cb.best_epoch} | val_acc tốt nhất = {cb.best:.4f}')

hist_tf = {
    'train_loss': [float(v) for v in h_tf.history['loss']],
    'val_loss':   [float(v) for v in h_tf.history['val_loss']],
    'train_acc':  [float(v) for v in h_tf.history['accuracy']],
    'val_acc':    [float(v) for v in h_tf.history['val_accuracy']],
}
best_ep_tf = cb.best_epoch

## 7. Đánh giá hai mô hình trên tập kiểm thử

In [ ]:
def summarize(y_true, probs, loss, name):
    pred = probs.argmax(axis=1)
    acc = float((pred == y_true).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred, average='macro',
                                                       zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=list(range(10)))
    per_class = (cm.diagonal() / cm.sum(axis=1)).astype(float)
    conf = probs.max(axis=1)
    wrong = np.where(pred != y_true)[0]
    order = wrong[np.argsort(-conf[wrong])][:8]
    hce = [{'index': int(i), 'true': int(y_true[i]), 'pred': int(pred[i]),
            'confidence': float(conf[i])} for i in order]
    print(f'--- {name} trên {len(y_true):,} ảnh kiểm thử ---')
    print(f'  loss            = {loss:.6f}')
    print(f'  accuracy        = {acc:.6f}  ({acc*100:.2f}%)')
    print(f'  macro precision = {prec:.6f}')
    print(f'  macro recall    = {rec:.6f}')
    print(f'  macro F1        = {f1:.6f}')
    print(f'  số ảnh sai      = {len(wrong):,}')
    return dict(loss=float(loss), accuracy=acc, macro_precision=float(prec),
                macro_recall=float(rec), macro_f1=float(f1),
                confusion_matrix=cm.tolist(), per_class_accuracy=per_class.tolist(),
                high_conf_errors=hce, pred=pred, probs=probs)


loss_pt_te, acc_pt_te, probs_pt = eval_torch(model_pt, test_ld)
res_pt = summarize(y_te, probs_pt, loss_pt_te, 'PyTorch')
print()

logits_tf = model_tf.predict(X_te_k, batch_size=512, verbose=0)
probs_tf = tf.nn.softmax(logits_tf).numpy()
loss_tf_te = float(keras.losses.SparseCategoricalCrossentropy(from_logits=True)(
    y_te, logits_tf).numpy())
res_tf = summarize(y_te, probs_tf, loss_tf_te, 'Keras / TensorFlow')

print()
print('Chênh lệch PyTorch so với Keras:')
print(f'  accuracy : {(res_pt["accuracy"]-res_tf["accuracy"])*100:+.2f} điểm phần trăm')
print(f'  macro F1 : {(res_pt["macro_f1"]-res_tf["macro_f1"])*100:+.2f} điểm phần trăm')
print(f'  thời gian: {time_pt:.1f}s so với {time_tf:.1f}s '
      f'(tỉ lệ {time_tf/max(time_pt,1e-9):.2f}x)')
print()
print('Hai framework cùng kiến trúc, cùng siêu tham số, cùng hạt giống, nhưng khác nhau ở thứ tự')
print('sinh số ngẫu nhiên khi khởi tạo và khi xáo trộn lô, nên chênh lệch nhỏ là điều phải xảy ra.')
print('Chênh lệch này là thước đo thực nghiệm cho phương sai do ngẫu nhiên hóa: mọi khác biệt nhỏ')
print('hơn nó giữa các cấu hình đều không nên được diễn giải là có ý nghĩa.')

In [ ]:
best_name = 'PyTorch' if res_pt['accuracy'] >= res_tf['accuracy'] else 'Keras / TensorFlow'
res_best  = res_pt if res_pt['accuracy'] >= res_tf['accuracy'] else res_tf
print(f'Mô hình tốt nhất: {best_name}')
print()
print(f'Báo cáo phân loại chi tiết của {best_name} trên 10 000 ảnh kiểm thử:')
print(classification_report(y_te, res_best['pred'], digits=4, zero_division=0,
                            target_names=[f'{k}. {CLASS_VI[k]}' for k in range(10)]))

**Diễn giải bảng phân loại.** Bảng cho thấy cùng một trật tự khó dễ đã quan sát ở notebook 01
nhưng với biên độ được cải thiện trên toàn bộ mười lớp. Điểm đáng chú ý về phương pháp: `precision`
và `recall` của cùng một lớp không bằng nhau, và chênh lệch giữa chúng cho biết mô hình đang thiên
về phía nào. Lớp có `recall` thấp mà `precision` cao là lớp mà mô hình "ngại" dự đoán; ngược lại,
lớp có `recall` cao mà `precision` thấp là lớp mà mô hình dùng làm "thùng chứa" cho những ảnh khó.

## 8. Đường cong huấn luyện của hai framework

In [ ]:
ep_axis = np.arange(1, EPOCHS_FW + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_axis, hist_pt['train_loss'], 'o-',  color='#8E44AD', label='PyTorch train')
axes[0].plot(ep_axis, hist_pt['val_loss'],  'o--', color='#8E44AD', alpha=0.55, label='PyTorch val')
axes[0].plot(ep_axis, hist_tf['train_loss'], 's-',  color='#D35400', label='Keras train')
axes[0].plot(ep_axis, hist_tf['val_loss'],  's--', color='#D35400', alpha=0.55, label='Keras val')
axes[0].set_title('Mất mát entropy chéo theo epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Mất mát')
axes[0].legend(fontsize=9); axes[0].set_xticks(ep_axis)

axes[1].plot(ep_axis, np.array(hist_pt['train_acc'])*100, 'o-',  color='#8E44AD', label='PyTorch train')
axes[1].plot(ep_axis, np.array(hist_pt['val_acc'])*100,  'o--', color='#8E44AD', alpha=0.55, label='PyTorch val')
axes[1].plot(ep_axis, np.array(hist_tf['train_acc'])*100, 's-',  color='#D35400', label='Keras train')
axes[1].plot(ep_axis, np.array(hist_tf['val_acc'])*100,  's--', color='#D35400', alpha=0.55, label='Keras val')
axes[1].axvline(best_ep_pt, color='#8E44AD', ls=':', lw=1.2)
axes[1].axvline(best_ep_tf, color='#D35400', ls=':', lw=1.2)
axes[1].set_title('Độ chính xác theo epoch (đường chấm dọc = epoch tốt nhất)', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Độ chính xác (%)')
axes[1].legend(fontsize=9, loc='lower right'); axes[1].set_xticks(ep_axis)

fig.suptitle('CNN sâu ba khối trên CIFAR-10: PyTorch so với Keras\n'
             f'(huấn luyện đầy đủ {len(X_tr):,} ảnh train, {len(X_va):,} ảnh validation, '
             f'batch {BATCH_FW})', fontsize=14, y=1.04)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_framework_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_framework_curves.png')
print()
print(f'val_acc cuối     : PyTorch {hist_pt["val_acc"][-1]*100:.2f}% | '
      f'Keras {hist_tf["val_acc"][-1]*100:.2f}%')
print(f'val_acc tốt nhất : PyTorch {max(hist_pt["val_acc"])*100:.2f}% (epoch {best_ep_pt}) | '
      f'Keras {max(hist_tf["val_acc"])*100:.2f}% (epoch {best_ep_tf})')
print(f'train_acc cuối   : PyTorch {hist_pt["train_acc"][-1]*100:.2f}% | '
      f'Keras {hist_tf["train_acc"][-1]*100:.2f}%')
print(f'Khoảng cách train - val ở epoch cuối: '
      f'PyTorch {(hist_pt["train_acc"][-1]-hist_pt["val_acc"][-1])*100:+.2f} đpt | '
      f'Keras {(hist_tf["train_acc"][-1]-hist_tf["val_acc"][-1])*100:+.2f} đpt')

**Diễn giải hình `fig_cifar10_framework_curves.png`.** Hai framework vẽ ra hai đường gần như chồng
lên nhau, đúng như kỳ vọng khi kiến trúc, siêu tham số và thuật toán tối ưu đều giống nhau. Đây là
một kết quả có giá trị: nó xác nhận rằng lựa chọn framework **không** phải là yếu tố quyết định
chất lượng mô hình, và mọi so sánh giữa PyTorch và TensorFlow nên tập trung vào công thái học lập
trình và hệ sinh thái triển khai chứ không phải vào độ chính xác.

So sánh với đường cong của mạng NumPy ở notebook 01, khoảng cách giữa đường train và đường
validation ở đây hẹp hơn đáng kể mặc dù mạng có nhiều tham số hơn. Dropout và BatchNormalization
đã làm đúng nhiệm vụ chính quy hóa của chúng. Thêm nữa, các đường ở đây được vẽ trên 40 000 ảnh
huấn luyện thay vì 10 000, và nhiều dữ liệu hơn tự nó đã là biện pháp chống quá khớp hiệu quả nhất.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
tick_lbl = [f'{k}\n{CLASS_VI[k]}' for k in range(10)]
for ax, res, name in [(axes[0], res_pt, 'PyTorch'), (axes[1], res_tf, 'Keras / TensorFlow')]:
    cm = np.array(res['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar=False,
                annot_kws={'size': 7.5}, linewidths=0.4, linecolor='#DDDDDD',
                xticklabels=tick_lbl, yticklabels=[f'{k}. {CLASS_VI[k]}' for k in range(10)])
    ax.set_title(f'{name}: accuracy = {res["accuracy"]*100:.2f}%, '
                 f'{int(cm.sum() - np.trace(cm)):,} ảnh sai', fontsize=12)
    ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thật')
    ax.tick_params(axis='x', labelsize=8); ax.tick_params(axis='y', labelsize=8)
fig.suptitle('Ma trận nhầm lẫn trên 10 000 ảnh kiểm thử CIFAR-10, CNN sâu hai framework',
             fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_framework_confusion.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_framework_confusion.png')

for tag, res in [('PyTorch', res_pt), ('Keras', res_tf)]:
    cm = np.array(res['confusion_matrix'])
    off = sorted([(cm[a, b], a, b) for a in range(10) for b in range(10)
                  if a != b and cm[a, b] > 0], reverse=True)[:5]
    print(f'\nNăm cặp nhầm lẫn nặng nhất của {tag}:')
    for n, a, b in off:
        print(f'  {CLASS_VI[a]:>9s} -> {CLASS_VI[b]:<9s} : {n:>4} ảnh '
              f'({100*n/cm[a].sum():5.2f}% số ảnh lớp {CLASS_VI[a]})')

cm_b = np.array(res_best['confusion_matrix'])
veh, ani = [0, 1, 8, 9], [2, 3, 4, 5, 6, 7]
n_err = int(cm_b.sum() - np.trace(cm_b))
err_veh = int(cm_b[np.ix_(veh, veh)].sum() - sum(cm_b[k, k] for k in veh))
err_ani = int(cm_b[np.ix_(ani, ani)].sum() - sum(cm_b[k, k] for k in ani))
print()
print(f'Phân rã lỗi của mô hình tốt nhất ({best_name}) theo nhóm ngữ nghĩa:')
print(f'  tổng số ảnh sai                   : {n_err:,}')
print(f'  sai trong nội bộ nhóm phương tiện : {err_veh:,} ({100*err_veh/n_err:.1f}%)')
print(f'  sai trong nội bộ nhóm động vật    : {err_ani:,} ({100*err_ani/n_err:.1f}%)')
print(f'  sai chéo giữa hai nhóm            : {n_err-err_veh-err_ani:,} '
      f'({100*(n_err-err_veh-err_ani)/n_err:.1f}%)')

**Diễn giải hình `fig_cifar10_framework_confusion.png`.** Hai ma trận có cùng cấu trúc lỗi, một
bằng chứng nữa cho thấy hai framework học ra cùng một thứ. Khối lượng ngoài đường chéo tập trung
ở các ô nối những lớp gần nhau về mặt thị giác, và phép phân rã theo nhóm ngữ nghĩa ở ô trên cho
thấy phần lớn lỗi nằm **trong** nhóm chứ không **giữa** hai nhóm.

Điều này khớp với trực giác về thứ tự học của mạng phân cấp: sự khác biệt giữa một khối kim loại
có bánh xe và một sinh vật có bốn chân được mã hóa ở những đặc trưng thô, dễ học; còn sự khác biệt
giữa mèo và chó nằm ở kết cấu lông và tỉ lệ khuôn mặt, những thứ gần như biến mất ở độ phân giải
$32 \times 32$.

## 9. Đối chiếu ba cách cài đặt

In [ ]:
with open(os.path.join(REP_DIR, 'metrics_cifar10_scratch_partial.json'), encoding='utf-8') as f:
    partial = json.load(f)
np_base, np_impr = partial['numpy_baseline'], partial['numpy_improved']
sub = partial['subset']
EPOCHS_NP = int(sub['epochs'])
print('Nạp lại kết quả NumPy từ notebook 01:')
print(f"  numpy_baseline : acc={np_base['accuracy']*100:.2f}% macroF1={np_base['macro_f1']*100:.2f}% "
      f"params={np_base['params']:,} time={np_base['train_time_s']:.1f}s")
print(f"  numpy_improved : acc={np_impr['accuracy']*100:.2f}% macroF1={np_impr['macro_f1']*100:.2f}% "
      f"params={np_impr['params']:,} time={np_impr['train_time_s']:.1f}s")
print(f"  tập con huấn luyện NumPy: {sub['n_train_subset']:,} train / {sub['n_val_subset']:,} val")
print(f"  kiểm chứng gradient: {partial['gradient_check']['n_checks']} phép, "
      f"sai số tương đối lớn nhất = {partial['gradient_check']['max_rel_error']:.3e}")

In [ ]:
names   = ['NumPy (Improved)', 'PyTorch', 'Keras / TensorFlow']
metrics_lbl = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']
table = np.array([
    [np_impr['accuracy'], np_impr['macro_precision'], np_impr['macro_recall'], np_impr['macro_f1']],
    [res_pt['accuracy'],  res_pt['macro_precision'],  res_pt['macro_recall'],  res_pt['macro_f1']],
    [res_tf['accuracy'],  res_tf['macro_precision'],  res_tf['macro_recall'],  res_tf['macro_f1']],
]) * 100

xpos = np.arange(len(metrics_lbl)); w = 0.26
colors = ['#C0392B', '#8E44AD', '#D35400']
fig, ax = plt.subplots(figsize=(12.5, 6.4))
for i, nm in enumerate(names):
    bars = ax.bar(xpos + (i - 1) * w, table[i], w, label=nm,
                  color=colors[i], edgecolor='black', linewidth=0.6)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{bar.get_height():.2f}', ha='center', fontsize=8.5)
ax.axhline(10.0, color='gray', ls='-.', lw=1.2, label='Mốc đoán ngẫu nhiên 10%')
ax.set_xticks(xpos); ax.set_xticklabels(metrics_lbl)
ax.set_ylabel('Giá trị (%)')
ax.set_ylim(0, table.max() + 12)
ax.set_title('CIFAR-10: đối chiếu ba cách cài đặt CNN 2D trên 10 000 ảnh kiểm thử\n'
             f'(NumPy huấn luyện trên tập con {sub["n_train_subset"]:,} ảnh; '
             f'PyTorch và Keras trên đủ {len(X_tr):,} ảnh)', fontsize=13)
ax.legend(fontsize=9.5, loc='upper right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_3way_benchmark.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_3way_benchmark.png')
print()
hdr = (f"{'Cách cài đặt':<20}{'Accuracy':>11}{'MacroP':>10}{'MacroR':>10}{'MacroF1':>10}"
       f"{'Tham số':>12}{'Thời gian':>12}{'Ảnh train':>11}")
print(hdr); print('-' * len(hdr))
rows_meta = [(names[0], np_impr['params'], np_impr['train_time_s'], sub['n_train_subset']),
             (names[1], n_pt, time_pt, len(X_tr)),
             (names[2], n_tf, time_tf, len(X_tr))]
for i, (nm, pr, tt, ntr) in enumerate(rows_meta):
    print(f'{nm:<20}{table[i,0]:>10.2f}%{table[i,1]:>9.2f}%{table[i,2]:>9.2f}%'
          f'{table[i,3]:>9.2f}%{pr:>12,}{tt:>11.1f}s{ntr:>11,}')
print('-' * len(hdr))
print(f'Baseline NumPy (tham chiếu): accuracy {np_base["accuracy"]*100:.2f}%, '
      f'macro F1 {np_base["macro_f1"]*100:.2f}%, {np_base["params"]:,} tham số')
print()
gap = (max(res_pt['accuracy'], res_tf['accuracy']) - np_impr['accuracy']) * 100
print(f'Khoảng cách giữa mô hình framework tốt nhất và NumPy Improved: {gap:.2f} điểm phần trăm')

**Diễn giải hình `fig_cifar10_3way_benchmark.png`.** Khoảng cách giữa nhóm framework và cài đặt
NumPy được ghi bằng số cụ thể ở ô trên. Báo cáo **không** quy toàn bộ khoảng cách đó cho chất
lượng cài đặt, vì ba yếu tố cùng khác nhau:

1. **Lượng dữ liệu huấn luyện.** 40 000 ảnh so với 10 000 ảnh, chênh nhau bốn lần. Trên CIFAR-10,
   đây gần như chắc chắn là yếu tố lớn nhất.
2. **Chính quy hóa.** Mạng framework có Dropout và BatchNorm; mạng NumPy không có gì. Notebook 01
   đã ghi nhận khoảng cách train - val của mạng NumPy rộng hơn rõ rệt.
3. **Chiều sâu.** Ba khối tích chập so với hai, kèm trường tiếp nhận rộng hơn.

Điều mà phép so sánh này **có** chứng minh: cài đặt NumPy không sai, vì nó vượt xa mốc ngẫu nhiên
10% và bám theo cùng một trật tự khó dễ giữa các lớp như hai framework. Cài đặt thủ công có giá
trị sư phạm ở chỗ nó phơi bày toàn bộ phép toán; framework có giá trị thực tiễn ở chỗ nó cho phép
thử nghiệm kiến trúc phức tạp trong thời gian chấp nhận được.

Về thời gian chạy, con số đo được ở bảng phản ánh một sự thật thường bị bỏ qua: trên CPU, ưu thế
tốc độ của framework so với NumPy thuần không lớn như trên GPU, vì cả hai cuối cùng đều gọi xuống
cùng những thư viện BLAS. Framework thắng ở chỗ khác — ở khả năng biểu đạt kiến trúc và ở phép vi
phân tự động.

## 10. Phân tích sâu mô hình tốt nhất

In [ ]:
pca_ = np.array(res_best['per_class_accuracy']) * 100
support = np.bincount(y_te, minlength=10)
order_worst = np.argsort(pca_)

fig, ax = plt.subplots(figsize=(12.5, 6.4))
cols = ['#27AE60' if v >= pca_.mean() else '#E74C3C' for v in pca_]
bars = ax.bar(np.arange(10), pca_, color=cols, edgecolor='black', linewidth=0.6)
ax.axhline(pca_.mean(), color='navy', ls='--', lw=1.4,
           label=f'Trung bình theo lớp = {pca_.mean():.2f}%')
ax.axhline(res_best['accuracy']*100, color='darkorange', ls=':', lw=1.6,
           label=f'Độ chính xác tổng thể = {res_best["accuracy"]*100:.2f}%')
for k, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pca_[k]:.2f}', ha='center', fontsize=9)
ax.set_xticks(np.arange(10))
ax.set_xticklabels([f'{k}\n{CLASS_VI[k]}' for k in range(10)], fontsize=9)
ax.set_xlabel('Lớp'); ax.set_ylabel('Độ chính xác (%)')
ax.set_ylim(0, 105)
for k, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, 2.0,
            f'n={support[k]}', ha='center', va='bottom', fontsize=8, color='white')
ax.set_title(f'Độ chính xác theo từng lớp của mô hình tốt nhất ({best_name})\n'
             'trên 10 000 ảnh kiểm thử CIFAR-10', fontsize=13)
ax.legend(fontsize=9.5, loc='lower right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_per_class_accuracy.png')
print()
print(f"{'Lớp':>12}{'Accuracy':>11}{'Số ảnh':>9}{'Số ảnh sai':>13}")
print('-' * 46)
cmb = np.array(res_best['confusion_matrix'])
for k in range(10):
    print(f'{CLASS_VI[k]:>12}{pca_[k]:>10.2f}%{support[k]:>9}{int(support[k]-cmb[k,k]):>13}')
print('-' * 46)
print(f'Lớp tốt nhất : {CLASS_VI[int(pca_.argmax())]} với {pca_.max():.2f}%')
print(f'Lớp kém nhất : {CLASS_VI[int(pca_.argmin())]} với {pca_.min():.2f}%')
print(f'Biên độ dao động giữa lớp tốt nhất và kém nhất: {pca_.max()-pca_.min():.2f} điểm phần trăm')
print('Ba lớp khó nhất theo thứ tự:',
      ', '.join(f'{CLASS_VI[int(k)]} ({pca_[k]:.2f}%)' for k in order_worst[:3]))
print()
veh_acc = pca_[[0, 1, 8, 9]].mean()
ani_acc = pca_[[2, 3, 4, 5, 6, 7]].mean()
print(f'Độ chính xác trung bình nhóm phương tiện (máy bay, ô tô, tàu thủy, xe tải): {veh_acc:.2f}%')
print(f'Độ chính xác trung bình nhóm động vật (chim, mèo, hươu, chó, ếch, ngựa)   : {ani_acc:.2f}%')
print(f'Chênh lệch giữa hai nhóm: {veh_acc-ani_acc:+.2f} điểm phần trăm')

**Diễn giải hình `fig_cifar10_per_class_accuracy.png`.** Biên độ dao động giữa lớp dễ nhất và lớp
khó nhất lớn hơn rất nhiều so với MNIST, nơi mọi chữ số đều nằm trong một dải hẹp. Sự phân tầng
này không ngẫu nhiên mà bám theo đúng ranh giới ngữ nghĩa: nhóm phương tiện đạt trung bình cao hơn
nhóm động vật một khoảng được ghi rõ ở ô trên.

Lý do nằm ở bản chất của đối tượng. Phương tiện là vật thể cứng, có cạnh thẳng, tỉ lệ ổn định, và
thường xuất hiện trên một loại nền đặc trưng. Động vật là vật thể biến dạng, xuất hiện ở vô số tư
thế, và nhiều lớp trong nhóm này chia sẻ cùng một sơ đồ hình thể bốn chân. Mọi số liệu trong mục
này đều nhất quán với phân tích định tính đã nêu ở notebook 00 trước khi có bất kỳ mô hình nào
được huấn luyện.

In [ ]:
hce = res_best['high_conf_errors']
fig, axes = plt.subplots(2, 4, figsize=(13.5, 7.6))
for ax, item in zip(axes.ravel(), hce):
    i = item['index']
    ax.imshow(x_test_raw[i])
    ax.set_title(f"#{i} · thật = {CLASS_VI[item['true']]} · dự đoán = {CLASS_VI[item['pred']]}\n"
                 f"độ tin cậy = {item['confidence']*100:.2f}%", fontsize=10.5, color='#B03A2E')
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.ravel()[len(hce):]:
    ax.axis('off')
fig.suptitle(f'Tám ảnh bị phân loại sai với độ tin cậy cao nhất ({best_name})\n'
             'trên 10 000 ảnh kiểm thử CIFAR-10', fontsize=14, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f'{FIG_DIR}/fig_cifar10_high_conf_errors.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_high_conf_errors.png')
print()
print(f"{'STT':>4}{'Chỉ số ảnh':>12}{'Nhãn thật':>12}{'Dự đoán':>12}{'Độ tin cậy':>13}")
print('-' * 55)
for r, item in enumerate(hce, 1):
    print(f"{r:>4}{item['index']:>12}{CLASS_VI[item['true']]:>12}{CLASS_VI[item['pred']]:>12}"
          f"{item['confidence']*100:>12.2f}%")
print('-' * 55)
conf_all = res_best['probs'].max(axis=1)
pred_all = res_best['pred']
wrong_mask = pred_all != y_te
print(f'Độ tin cậy trung bình khi dự đoán ĐÚNG : {conf_all[~wrong_mask].mean()*100:.2f}%')
print(f'Độ tin cậy trung bình khi dự đoán SAI  : {conf_all[wrong_mask].mean()*100:.2f}%')
print(f'Số ảnh sai có độ tin cậy > 99%         : {int(((conf_all > 0.99) & wrong_mask).sum()):,}')
print(f'Tổng số ảnh sai                        : {int(wrong_mask.sum()):,}')
print(f'Tỉ lệ ảnh sai nhưng rất tự tin (>99%)  : '
      f'{100*((conf_all > 0.99) & wrong_mask).sum()/max(1, wrong_mask.sum()):.2f}% số ảnh sai')

**Diễn giải hình `fig_cifar10_high_conf_errors.png`.** Tám ảnh này là những trường hợp mô hình sai
mà vẫn gán xác suất rất cao cho nhãn sai — loại lỗi nguy hiểm nhất trong thực tế, vì độ tin cậy
cao thường được dùng làm căn cứ để bỏ qua khâu kiểm tra của con người.

Quan sát trực tiếp trên ảnh cho thấy phần lớn thuộc ba dạng: đối tượng bị cắt cụt hoặc chỉ thấy
một phần, đối tượng nhỏ trên nền chiếm ưu thế, và các cặp lớp gần nhau về hình thể. Một số ảnh khó
tới mức người xem cũng phải do dự, điều này phản ánh giới hạn thực sự của độ phân giải
$32 \times 32$ chứ không phải khiếm khuyết riêng của mô hình.

Hai con số ở cuối ô mã có ý nghĩa về hiệu chuẩn xác suất: độ tin cậy trung bình khi đúng cao hơn
rõ rệt so với khi sai, nghĩa là điểm tin cậy của mô hình **có** mang thông tin và dùng được làm
tín hiệu lọc. Tuy nhiên tỉ lệ ảnh sai mà vẫn tự tin trên 99% khác 0, nên ngưỡng tin cậy không bao
giờ là bảo đảm tuyệt đối.

## 11. Ghi tệp metrics tổng hợp

In [ ]:
def pack_fw(res, hist, best_ep, ttime, n_params, framework, epochs):
    return {
        'framework': framework,
        'params': int(n_params),
        'train_time_s': float(ttime),
        'epochs': int(epochs),
        'best_epoch': int(best_ep),
        'accuracy': res['accuracy'],
        'macro_precision': res['macro_precision'],
        'macro_recall': res['macro_recall'],
        'macro_f1': res['macro_f1'],
        'loss': res['loss'],
        'history': {k: [float(v) for v in hist[k]]
                    for k in ('train_loss', 'val_loss', 'train_acc', 'val_acc')},
        'confusion_matrix': res['confusion_matrix'],
        'per_class_accuracy': res['per_class_accuracy'],
        'high_conf_errors': res['high_conf_errors'],
    }


notes = (
    'Hai mô hình NumPy thuần (numpy_baseline, numpy_improved) được huấn luyện trên TẬP CON '
    f'PHÂN TẦNG {sub["n_train_subset"]} ảnh train và {sub["n_val_subset"]} ảnh validation '
    f'({EPOCHS_NP} epoch, batch {sub["batch_size"]}) thay vì toàn bộ 40000/10000, do mạng tích '
    'chập cài bằng NumPy thuần cho ảnh màu 32x32x3 chạy trên CPU vượt ngân sách 15 phút mỗi '
    'notebook mà hợp đồng quy định; chi phí của ảnh ba kênh 32x32 lớn hơn nhiều lần so với ảnh '
    'xám 28x28 của miền mnist. Hai mô hình framework (pytorch, tensorflow) dùng ĐẦY ĐỦ 40000 ảnh '
    f'train và 10000 ảnh validation, {EPOCHS_FW} epoch, batch {BATCH_FW}. CẢ BỐN mô hình đều được '
    'đánh giá trên trọn vẹn 10000 ảnh của tập kiểm thử gốc nên phép so sánh công bằng ở phía đánh '
    'giá; chênh lệch giữa nhóm NumPy và nhóm framework là tổng hợp của ba yếu tố: lượng dữ liệu '
    'huấn luyện (gấp 4 lần), chính quy hóa (framework có Dropout và BatchNorm, NumPy không có) và '
    'chiều sâu (3 khối so với 2). Khóa roc_auc của schema nhị phân không áp dụng cho bài toán 10 '
    'lớp nên được lược bỏ, thay bằng macro_precision / macro_recall / macro_f1 / '
    'per_class_accuracy / high_conf_errors đúng theo biến thể đa lớp của mục 5.3. Kiểm chứng '
    'gradient bằng sai phân hữu hạn trung tâm (eps=1e-5, float64) trên '
    f'{partial["gradient_check"]["n_checks"]} vị trí tham số của mạng BA KÊNH cho sai số tương đối '
    f'lớn nhất {partial["gradient_check"]["max_rel_error"]:.3e}, xác nhận phép lan truyền ngược '
    'viết tay xử lý đúng chiều kênh màu. Chuẩn hóa dùng ba cặp hằng số riêng cho ba kênh R, G, B '
    'thay vì một cặp vô hướng như miền mnist. PyTorch được đặt torch.set_num_threads(4) vì đo đạc '
    'trên máy này cho thấy cấu hình mặc định chậm hơn nhiều lần. Notebook 01 còn ghi tệp trung '
    'gian reports/metrics_cifar10_scratch_partial.json chứa nhật ký chi tiết của kiểm chứng '
    'gradient; tệp đó bổ sung chứ không thay thế tệp bắt buộc này.'
)

metrics = {
    'domain': 'cifar10',
    'task': 'classification',
    'dataset': {
        'file': 'cifar10/data/cifar10.npz',
        'n_raw': int(len(x_train_raw) + len(x_test_raw)),
        'n_clean': int(len(x_train_raw) + len(x_test_raw)),
        'n_train': int(len(X_tr)),
        'n_val': int(len(X_va)),
        'n_test': int(len(X_te)),
        'n_features': 3072,
        'image_shape': [32, 32, 3],
        'n_classes': 10,
        'class_names_en': CLASS_EN,
        'class_names_vi': CLASS_VI,
        'preprocess': {'mean': [float(v) for v in MEAN_C],
                       'std': [float(v) for v in STD_C], 'scale': 255.0},
        'numpy_subset': {'n_train': int(sub['n_train_subset']),
                         'n_val': int(sub['n_val_subset'])},
    },
    'numpy_subset': {'n_train': int(sub['n_train_subset']),
                     'n_val': int(sub['n_val_subset'])},
    'models': {
        'numpy_baseline': {k: v for k, v in np_base.items()},
        'numpy_improved': {k: v for k, v in np_impr.items()},
        'pytorch':    pack_fw(res_pt, hist_pt, best_ep_pt, time_pt, n_pt,
                              'PyTorch 2.9.1 (CPU)', EPOCHS_FW),
        'tensorflow': pack_fw(res_tf, hist_tf, best_ep_tf, time_tf, n_tf,
                              'TensorFlow 2.21.0 / Keras 3.15.1 (CPU)', EPOCHS_FW),
    },
    'best_model': 'pytorch' if res_pt['accuracy'] >= res_tf['accuracy'] else 'tensorflow',
    'gradient_check': partial['gradient_check'],
    'notes': notes,
}

out = os.path.join(REP_DIR, 'metrics_cifar10.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('Đã ghi', out, f'({os.path.getsize(out)/1024:.1f} KB)')
print()
for key, m in metrics['models'].items():
    assert len(m['confusion_matrix']) == 10 and len(m['confusion_matrix'][0]) == 10
    assert len(m['per_class_accuracy']) == 10
    assert len(m['high_conf_errors']) <= 8
    print(f"  {key:16s} acc={m['accuracy']:.6f} macroF1={m['macro_f1']:.6f} "
          f"params={m['params']:>9,} time={m['train_time_s']:8.1f}s "
          f"cfm=10x10 per_class=10 hce={len(m['high_conf_errors'])}")
print()
print('Mô hình tốt nhất ghi trong metrics :', metrics['best_model'])
print('Khối numpy_subset ở cấp cao nhất   :', metrics['numpy_subset'])
print('Khối gradient_check ở cấp cao nhất :',
      f"{metrics['gradient_check']['n_checks']} phép kiểm, "
      f"sai số lớn nhất {metrics['gradient_check']['max_rel_error']:.3e}")

In [ ]:
required = [
    'fig_cifar10_class_distribution.png', 'fig_cifar10_sample_grid.png',
    'fig_cifar10_scratch_curves.png', 'fig_cifar10_scratch_confusion.png',
    'fig_cifar10_scratch_comparison.png', 'fig_cifar10_framework_curves.png',
    'fig_cifar10_framework_confusion.png', 'fig_cifar10_3way_benchmark.png',
    'fig_cifar10_per_class_accuracy.png', 'fig_cifar10_high_conf_errors.png',
]
print('Kiểm tra đủ 10 hình bắt buộc theo mục 6 hợp đồng tích hợp:')
ok = True
for f in required:
    p = os.path.join(FIG_DIR, f)
    e = os.path.exists(p)
    ok &= e
    print(f"  {'OK   ' if e else 'THIẾU'} {f:42s} "
          f"{os.path.getsize(p)/1024 if e else 0:8.1f} KB")
print()
print('Đủ 10/10 hình:', ok)
print()
print('Hiện vật mô hình bàn giao cho mlp_vs_cnn:')
for f in ['cifar10_cnn_pytorch.pt', 'cifar10_cnn_def.py', 'cifar10_preproc.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f:26s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')
print()
print('Tệp metrics:')
for f in ['metrics_cifar10.json', 'metrics_cifar10_scratch_partial.json',
          'cifar10_eda_summary.json']:
    p = os.path.join(REP_DIR, f)
    print(f'  {f:40s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')

## 12. Kết luận của miền CIFAR-10

Ba notebook của miền này đã hoàn thành trọn vẹn yêu cầu của đề bài đối với dữ liệu ảnh màu: cùng
một bài toán được giải bằng ba cách cài đặt độc lập, trên cùng một phép chia dữ liệu và cùng một
phép chuẩn hóa, rồi được đối chiếu trên cùng 10 000 ảnh kiểm thử.

**Về mặt kỹ thuật**, báo cáo đã chứng minh mạng tích chập viết tay xử lý đúng ảnh ba kênh, thông
qua kiểm chứng gradient bằng sai phân hữu hạn ở độ chính xác `float64` và qua việc đối chiếu với
một cài đặt tham chiếu viết bằng vòng lặp tường minh. Đây là điều kiện cần trước khi bất kỳ con số
độ chính xác nào được coi là có ý nghĩa.

**Về mặt thực nghiệm**, thứ tự kết quả nhất quán với kỳ vọng lý thuyết: Baseline thấp nhất,
Improved cao hơn nhờ gói đệm biên cộng He Normal cộng lịch learning rate, hai framework cao nhất
nhờ kiến trúc sâu hơn, có chính quy hóa và được huấn luyện trên gấp bốn lần dữ liệu. Hai framework
gần như ngang nhau, xác nhận rằng lựa chọn công cụ không quyết định chất lượng.

**Về mặt so sánh liên miền**, khoảng cách giữa kết quả CIFAR-10 và kết quả MNIST của cùng những
kiến trúc này chính là đóng góp phân tích của miền dữ liệu này cho báo cáo tổng thể: nó cho thấy
kết luận "CNN giải quyết tốt bài toán ảnh" cần được phát biểu thận trọng hơn, gắn với độ phức tạp
thị giác của dữ liệu cụ thể.

Ba hiện vật mô hình đã được lưu và kiểm chứng quy trình nạp lại, sẵn sàng cho notebook
`mlp_vs_cnn` phân tích không gian ẩn 128 chiều bằng PCA.